# Trabajo Práctico N°2 — Preprocesamiento

**Alumno:** Gonzalo Zarazaga

---

Este notebook aplica las decisiones de preprocesamiento definidas durante el EDA (`02_eda.ipynb`).
El mismo pipeline se aplica tanto al dataset de entrenamiento como al dataset de predicción,
garantizando que el modelo vea exactamente el mismo formato de datos en ambos contextos.

**Regla fundamental:** todo lo que se haga sobre los datos de entrenamiento debe replicarse
exactamente sobre los datos de predicción, usando los parámetros derivados del conjunto de entrenamiento.

**Insumos:**
- `data/raw/smoking_prediction.csv` — conjunto de entrenamiento (50.000 filas, etiquetado)
- `data/raw/smoking_prediction_entrega.csv` — conjunto de predicción (5.692 filas, sin etiquetar)

**Salidas:**
- `data/processed/smoking_train_processed.csv` — datos de entrenamiento listos para modelar
- `data/processed/smoking_predict_processed.csv` — datos de predicción listos para inferencia
- `models/categorical_mappings.json` — mapeos de codificación para reproducibilidad

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

PATH_TRAIN   = "../data/raw/smoking_prediction.csv"
PATH_PREDICT = "../data/raw/smoking_prediction_entrega.csv"
PATH_PROC    = Path("../data/processed")
PATH_MODELS  = Path("../models")

PATH_PROC.mkdir(exist_ok=True)
PATH_MODELS.mkdir(exist_ok=True)

print("Carpetas listas:")
print(f"  {PATH_PROC.resolve()}")
print(f"  {PATH_MODELS.resolve()}")

## 1. Carga de los datasets

Se cargan ambos datasets en su estado original (sin ninguna transformación previa)
y se verifica que las columnas sean las esperadas.

In [ ]:
df_train   = pd.read_csv(PATH_TRAIN)
df_predict = pd.read_csv(PATH_PREDICT)

print(f"Dataset de entrenamiento: {df_train.shape[0]:,} filas × {df_train.shape[1]} columnas")
print(f"Dataset de predicción:    {df_predict.shape[0]:,} filas × {df_predict.shape[1]} columnas")

In [ ]:
# Verificar diferencias de columnas entre ambos datasets
cols_train   = set(df_train.columns)
cols_predict = set(df_predict.columns)

solo_en_train   = cols_train   - cols_predict
solo_en_predict = cols_predict - cols_train

print(f"Solo en entrenamiento (columna a predecir): {solo_en_train}")
print(f"Solo en predicción:                         {solo_en_predict}")
print(f"Columnas comunes: {len(cols_train & cols_predict)}")

# Confirmar que no hay nulos en ninguno
print(f"\nNulos en train:   {df_train.isna().sum().sum()}")
print(f"Nulos en predict: {df_predict.isna().sum().sum()}")

## 2. Estrategia de preprocesamiento

Basado en los hallazgos del EDA, se aplican las siguientes transformaciones:

| Acción | Variables | Motivo |
|---|---|---|
| **Eliminar** | `ID` (solo train) | Identificador técnico, no es una feature |
| **Eliminar** | `oral` (ambos) | Variable constante: único valor `"Y"` → sin varianza, sin poder predictivo |
| **Codificar** | `gender` | Categórica binaria → `M=1, F=0` |
| **Codificar** | `tartar` | Categórica binaria → `Y=1, N=0` |
| **No imputar** | todas | Sin valores nulos en ninguno de los dos datasets |
| **No tratar outliers** | numéricas | XGBoost es robusto a valores atípicos |
| **No escalar** | numéricas | XGBoost no requiere normalización |

> El dataset de predicción conserva `ID` para poder asociar cada predicción con su registro original.

## 3. Eliminación de columnas irrelevantes

In [ ]:
COLS_ELIMINAR = ["oral"]  # constante en ambos datasets

# Entrenamiento: se elimina ID (no se necesita para modelar) y oral
df_train_proc = df_train.drop(columns=["ID"] + COLS_ELIMINAR)

# Predicción: se conserva ID para la entrega final, solo se elimina oral
df_predict_proc = df_predict.drop(columns=COLS_ELIMINAR)

print(f"Entrenamiento — antes: {df_train.shape}  →  después: {df_train_proc.shape}")
print(f"Predicción    — antes: {df_predict.shape}  →  después: {df_predict_proc.shape}")

print(f"\nColumnas eliminadas de entrenamiento: {['ID'] + COLS_ELIMINAR}")
print(f"Columnas eliminadas de predicción:    {COLS_ELIMINAR}")

## 4. Codificación de variables categóricas

Las variables `gender` y `tartar` son categóricas binarias. Se convierten a valores numéricos
usando mapeos deterministas. El mapeo se define a partir del conjunto de entrenamiento
y se aplica de forma idéntica al conjunto de predicción.

| Variable | Original | Codificado |
|---|---|---|
| `gender` | `F` | `0` |
| `gender` | `M` | `1` |
| `tartar` | `N` | `0` |
| `tartar` | `Y` | `1` |

In [ ]:
# Mapeos definidos sobre los valores observados en el conjunto de entrenamiento
MAPEOS_CATEGORICOS = {
    "gender": {"F": 0, "M": 1},
    "tartar": {"N": 0, "Y": 1}
}

for col, mapeo in MAPEOS_CATEGORICOS.items():
    df_train_proc[col]   = df_train_proc[col].map(mapeo)
    df_predict_proc[col] = df_predict_proc[col].map(mapeo)

# Verificación: no deben quedar valores nulos (indicaría un valor no visto en entrenamiento)
print("Verificación post-codificación:")
for col in MAPEOS_CATEGORICOS:
    nulos_train   = df_train_proc[col].isna().sum()
    nulos_predict = df_predict_proc[col].isna().sum()
    vals_train    = sorted(df_train_proc[col].unique())
    vals_predict  = sorted(df_predict_proc[col].unique())
    print(f"  '{col}' — train: {vals_train} (nulos: {nulos_train}) | predict: {vals_predict} (nulos: {nulos_predict})")

## 5. Verificación del dataset procesado

In [ ]:
print("=== Dataset de entrenamiento procesado ===")
df_train_proc.info()
print(f"\nNulos totales: {df_train_proc.isna().sum().sum()}")
print(f"Columnas tipo object: {list(df_train_proc.select_dtypes('object').columns)} (debe ser vacío)")

In [ ]:
df_train_proc.head(5)

In [ ]:
print("=== Dataset de predicción procesado ===")
print(f"Shape: {df_predict_proc.shape}")
print(f"Nulos totales: {df_predict_proc.isna().sum().sum()}")
df_predict_proc.head(5)

## 6. Separación de features y target

Se separan las variables predictoras (`X`) de la variable objetivo (`y`) en el conjunto de entrenamiento.
XGBoost requiere que `y` sea numérico — la columna `smoking` ya es `0`/`1`.

In [ ]:
TARGET = "smoking"

X_train = df_train_proc.drop(columns=[TARGET])
y_train = df_train_proc[TARGET]

print(f"X_train: {X_train.shape}  ← features")
print(f"y_train: {y_train.shape}  ← target")
print(f"\ny_train dtype: {y_train.dtype}  (XGBoost requiere numérico: OK)")
print(f"\nDistribución del target:")
print(y_train.value_counts().rename({0: 'No fumador (0)', 1: 'Fumador (1)'}).to_string())
print(f"\nColumnas de features ({len(X_train.columns)}):")
for col in X_train.columns:
    print(f"  {col:<30} {X_train[col].dtype}")

## 7. Exportación de archivos procesados

Se exportan los datasets procesados a `data/processed/` y los artefactos del pipeline a `models/`.

| Archivo | Contenido |
|---|---|
| `data/processed/smoking_train_processed.csv` | Features + target (`smoking`), sin `ID` ni `oral` |
| `data/processed/smoking_predict_processed.csv` | Features + `ID`, sin `oral` ni `smoking` |
| `models/categorical_mappings.json` | Mapeos de codificación para reproducir el pipeline |

In [ ]:
# Exportar datasets procesados
out_train   = PATH_PROC / "smoking_train_processed.csv"
out_predict = PATH_PROC / "smoking_predict_processed.csv"

df_train_proc.to_csv(out_train, index=False)
df_predict_proc.to_csv(out_predict, index=False)

print(f"✓ {out_train}")
print(f"  → {df_train_proc.shape[0]:,} filas × {df_train_proc.shape[1]} columnas")
print()
print(f"✓ {out_predict}")
print(f"  → {df_predict_proc.shape[0]:,} filas × {df_predict_proc.shape[1]} columnas")

In [ ]:
# Exportar mapeos de codificación
out_mappings = PATH_MODELS / "categorical_mappings.json"

with open(out_mappings, "w", encoding="utf-8") as f:
    json.dump(MAPEOS_CATEGORICOS, f, indent=2, ensure_ascii=False)

print(f"✓ {out_mappings}")
print()
print("Contenido del archivo:")
print(json.dumps(MAPEOS_CATEGORICOS, indent=2))
print()
print("Cómo usarlo en otros notebooks:")
print("  import json")
print("  with open('../models/categorical_mappings.json') as f:")
print("      mapeos = json.load(f)")
print("  for col, m in mapeos.items():")
print("      df[col] = df[col].map(m)")

## 8. Resumen del preprocesamiento

### Transformaciones aplicadas

| Paso | Acción | Train | Predict |
|---|---|---|---|
| Eliminar `ID` | Drop | ✓ | ✗ (conservado para entrega) |
| Eliminar `oral` | Drop | ✓ | ✓ |
| Codificar `gender` | M→1, F→0 | ✓ | ✓ |
| Codificar `tartar` | Y→1, N→0 | ✓ | ✓ |
| Imputación de nulos | — | No aplica | No aplica |
| Tratamiento de outliers | — | No aplica | No aplica |
| Escalado de variables | — | No aplica | No aplica |

### Resultado

| Dataset | Filas | Columnas | Tipo |
|---|---|---|---|
| `smoking_train_processed.csv` | 50.000 | 25 | Features + target |
| `smoking_predict_processed.csv` | 5.692 | 25 | ID + features |

### Próximo paso

El notebook `04_entrenamiento_y_optimizacion.ipynb` carga `smoking_train_processed.csv`,
divide en train/test con estratificación y entrena el modelo XGBoost.